In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%%bash
echo "1. Setting up local directories..."
mkdir -p /content/scripts
mkdir -p /content/xai
mkdir -p /content/checkpoints
mkdir -p /content/data/preprocessed
mkdir -p /content/output/gradcam_results

echo "2. Copying Training & XAI scripts..."
cp -r /content/drive/MyDrive/colab/denseNet121/scripts/*.py /content/scripts/
cp -r /content/drive/MyDrive/colab/denseNet121/xai/*.py /content/xai/

echo "3. Copying trained model checkpoint..."
cp /content/drive/MyDrive/colab/denseNet121/checkpoints/final_model_standard_cnn.pth /content/checkpoints/

echo "4. EXTRACTING PREPROCESSED DATA..."
# Unzips the images directly to the fast local SSD
unzip -q /content/drive/MyDrive/colab/denseNet121/preprocessed/preprocessed_384.zip -d /content/data/

# Copies the exact split indices so the XAI pipeline evaluates the correct test set
cp /content/drive/MyDrive/colab/denseNet121/preprocessed/split_indices.json /content/data/preprocessed/

echo "Local environment fully ready!"

1. Setting up local directories...
2. Copying Training & XAI scripts...
3. Copying trained model checkpoint...
4. EXTRACTING PREPROCESSED DATA...
Local environment fully ready!


In [3]:
!pip install pytest pandas seaborn scikit-learn

In [4]:
%%bash
cd /content/xai
export PYTHONPATH="/content/scripts:$PYTHONPATH"

echo "Running XAI Unit Tests..."
python -m pytest test_gradcam.py -v

Running XAI Unit Tests...
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/xai
plugins: langsmith-0.7.30, typeguard-4.5.1, anyio-4.13.0
collecting ... collected 80 items

test_gradcam.py::TestGradCAMOutput::test_output_shapes PASSED            [  1%]
test_gradcam.py::TestGradCAMOutput::test_cam_range PASSED                [  2%]
test_gradcam.py::TestGradCAMOutput::test_target_class_respected PASSED   [  3%]
test_gradcam.py::TestGradCAMOutput::test_single_sample PASSED            [  5%]
test_gradcam.py::TestGradCAMOutput::test_relu_applied PASSED             [  6%]
test_gradcam.py::TestGradCAMOutput::test_hooks_removed PASSED            [  7%]
test_gradcam.py::TestGradCAMOutput::test_invalid_input_dimension PASSED  [  8%]
test_gradcam.py::TestMapDescriptors::test_uniform_map_sci_near_zero PASSED [ 10%]
test_gradcam.py::TestMapDescript

In [5]:
!cd /content/xai && PYTHONPATH="/content/scripts:$PYTHONPATH" python -u gradcam_analysis.py


GRAD-CAM XAI PIPELINE -- DenseNet-121 Fungal Microscopy

Device: cuda
  GPU: Tesla T4

[1/8] Loading trained DenseNet-121 checkpoint...
  Loaded DenseNet121Classifier from final_model_standard_cnn.pth
  Parameters: 6,949,634 total, 6,949,634 trainable
  Classes: ['Candida albicans', 'Candida glabrata']
  Normalization mean: ['0.2369']
  Normalization std:  ['0.0074']

[2/8] Loading test split indices...
  Test set size: 821 patches

[3/8] Constructing test dataset (NPYPatchDataset)...
  Dataset length: 821

  Target layer: model.features.denseblock4
  (last DenseBlock before GAP -- Selvaraju et al. 2020, sec. 3)

[4/8] Running Grad-CAM inference...
  Target class: argmax per sample (None enforced)
  Inference complete. Accuracy: 818/821 (99.6%)

GRAD-CAM DESCRIPTOR SUMMARY
Class                Outcome          N        SCI mean±std          H mean±std
----------------------------------------------------------------------
Candida albicans     Correct        438   0.346 ± 0.071       11

In [6]:
%%bash
echo "Backing up results to Google Drive..."
mkdir -p /content/drive/MyDrive/colab/denseNet121/output/gradcam_results/
cp -r /content/output/gradcam_results/* /content/drive/MyDrive/colab/denseNet121/output/gradcam_results/

echo "Backup complete!"

Backing up results to Google Drive...
Backup complete!


cp: cannot stat '/content/output/gradcam_results/*': No such file or directory
